In [0]:
%pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")

if all([
    client_id,
    tenant_id,
    client_secret,
    storage_account_name,
    container_name
]):
    print("Variáveis carregadas com sucesso!")
else:
    raise ValueError("Erro ao carregar variáveis do .env")

In [0]:
print(storage_account_name)
print(container_name)

In [0]:
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
path_vendas_raw = base_path + "vendas_raw/2026/02/21/"

df_teste = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option(
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
        "OAuth"
    )
    .option(
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )
    .option(
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
        client_id
    )
    .option(
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
        client_secret
    )
    .option(
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    )
    .load(path_vendas_raw)
)

display(df_teste.select("path", "length", "modificationTime"))
print("Conexão com ADLS Gen2 realizada com sucesso.")
